# INPE CBERS-4A WPM — Scene Browser

Two-catalog strategy:
- **lgi-stac** (`dgi.inpe.br/lgi-stac`) — `CBERS4A_WPM_L4_DN`: has real `cloud_cover`, supports filter, separate bands only
- **BDC STAC** (`data.inpe.br/bdc/stac`) — `CB4A-WPM-L4-DN-1`: per-band COGs on direct HTTP (BAND0 = PAN 2m, BAND1–4 = Blue/Green/Red/NIR 8m), streamable via `/vsicurl`

Search flow: filter by cloud in lgi-stac → cross-ref to BDC L4-DN by path/row/date → export scene list JSON for `train_scripts/export_annotation_patches.py` (streams patch windows + SFIM pansharpening; no scene download).

Note: the `CB4A-WPM-PCA-FUSED-1` product was dropped — its effective detail is ~8m on a 2m grid (INPE's PCA fusion adds little PAN detail; verified 2026-07-05).

In [2]:
import os
import math
import urllib.request
from pathlib import Path
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from dotenv import load_dotenv
import requests

load_dotenv()

LGI_STAC_URL = 'https://www.dgi.inpe.br/lgi-stac'   # cloud_cover populated
BDC_STAC_URL = 'https://data.inpe.br/bdc/stac/v1/'  # per-band L4-DN COG assets

LGI_COLLECTION = 'CBERS4A_WPM_L4_DN'
BDC_COLLECTION = 'CB4A-WPM-L4-DN-1'

BBOX       = [-51, -31, -49, -29]
DATE_RANGE = '2016-01-01/2026-12-30'
MAX_CLOUD  = 10    # % — hard filter applied in lgi-stac query
MAX_ITEMS  = 100
PATH_ROWS  = [               # path/rows surviving QGIS Pampa-boundary trim (data/inpe/scene_footprints_kept.geojson)
    (205, 151), (205, 152),
    (206, 151), (206, 152), (206, 153),
    (207, 151), (207, 152), (207, 153), (207, 154), (207, 155),
    (208, 149), (208, 151), (208, 152), (208, 153), (208, 154), (208, 155),
    (209, 148), (209, 149), (209, 150), (209, 151), (209, 152), (209, 153),
    (210, 148), (210, 149), (210, 150), (210, 151), (210, 152), (210, 153),
    (211, 148), (211, 149), (211, 150), (211, 151), (211, 152),
    (212, 148), (212, 149), (212, 150), (212, 151), (212, 152),
    (213, 149), (213, 150), (213, 151), (213, 152),
    (214, 150), (214, 151),
]
OUT_DIR    = Path('data/inpe')

## Step 1 — Search by cloud cover

Query `lgi-stac` (`CBERS4A_WPM_L4_DN`) — the only catalog with real `cloud_cover`. No BDC calls yet.

In [3]:
date_start, date_end = DATE_RANGE.split('/')
DATETIME = f'{date_start}T00:00:00/{date_end}T23:59:59'


def lgi_search(query):
    """Paginated lgi-stac search. Returns all matching features —
    a single request silently truncates at `limit` (bit us at 110 matches)."""
    features, page = [], 1
    while True:
        payload = {
            'collections': [LGI_COLLECTION],
            'datetime':    DATETIME,
            'limit':       MAX_ITEMS,
            'page':        page,
            'query':       query,
        }
        if not PATH_ROWS:
            payload['bbox'] = BBOX   # bbox redundant when path/row given
        r = requests.post(f'{LGI_STAC_URL}/search', json=payload, timeout=30)
        r.raise_for_status()
        data = r.json()
        got = data.get('features', [])
        features += got
        matched = data.get('context', {}).get('matched')
        if not got or (matched is not None and len(features) >= matched):
            return features, matched
        page += 1


# NOTE: lgi-stac cloud_cover is quantized to 10% steps (0, 10, 20, ...).
# lte, not lt — strict lt would drop scenes at exactly MAX_CLOUD (site is inclusive).
base_query = {'cloud_cover': {'lte': MAX_CLOUD}}

if PATH_ROWS:
    # server-side path/row filter; one request per pair — a single query with
    # `in` lists would wrongly match cross-products (e.g. 206/114)
    lgi_features = []
    for path, row in PATH_ROWS:
        feats, matched = lgi_search({**base_query,
                                     'path': {'eq': int(path)},
                                     'row':  {'eq': int(row)}})
        print(f'path/row {path}/{row}: {len(feats)} scenes (server matched {matched})')
        lgi_features += feats
    lgi_features.sort(key=lambda f: f['properties']['datetime'], reverse=True)
else:
    lgi_features, matched = lgi_search(base_query)
    print(f'bbox query: {len(lgi_features)} scenes (server matched {matched})')

print(f'\n{len(lgi_features)} scenes  (cloud <= {MAX_CLOUD}%'
      + (f', path/row in {sorted(set(PATH_ROWS))}' if PATH_ROWS else '') + ')\n')
print(f'{"#":<4} {"ID":<40} {"Date"}       {"Path/Row"}  {"Cloud%"}')
print('-' * 76)
for i, feat in enumerate(lgi_features):
    p     = feat['properties']
    date  = p.get('datetime', '?')[:10]
    cloud = p.get('cloud_cover')
    cloud_str = f'{cloud:>5.1f}%' if cloud is not None else '    ?%'
    pr = f'{p.get("path", "?")}/{p.get("row", "?")}'
    print(f'{i:<4} {feat["id"]:<40} {date}   {pr:<9} {cloud_str}')

path/row 205/151: 12 scenes (server matched 12)
path/row 205/152: 7 scenes (server matched 7)
path/row 206/151: 12 scenes (server matched 12)
path/row 206/152: 10 scenes (server matched 10)
path/row 206/153: 8 scenes (server matched 8)
path/row 207/151: 12 scenes (server matched 12)
path/row 207/152: 11 scenes (server matched 11)
path/row 207/153: 7 scenes (server matched 7)
path/row 207/154: 8 scenes (server matched 8)
path/row 207/155: 11 scenes (server matched 11)
path/row 208/149: 15 scenes (server matched 15)
path/row 208/151: 14 scenes (server matched 14)
path/row 208/152: 12 scenes (server matched 12)
path/row 208/153: 14 scenes (server matched 14)
path/row 208/154: 9 scenes (server matched 9)
path/row 208/155: 4 scenes (server matched 4)
path/row 209/148: 17 scenes (server matched 17)
path/row 209/149: 12 scenes (server matched 12)
path/row 209/150: 16 scenes (server matched 16)
path/row 209/151: 18 scenes (server matched 18)
path/row 209/152: 14 scenes (server matched 14)
path

## Step 1c — Pick one scene per path/row, season-consistent

No single season covers all 42 path/rows (checked: summer 40/42, autumn 38/42, winter 41/42,
spring 41/42 — winter/spring tie for best). Pick a season, take the lowest-cloud scene per
path/row inside it; for the path/row(s) with nothing in-season, fall back to the
closest-by-month date and flag it as a logged exception (not silently widen the window).</cell id="a0000004">
<cell id="ecac5757">


In [4]:
# Southern-hemisphere seasons (by month). Winter/spring both cover 41/42 path/rows
# for this candidate set — pick one; re-check coverage if PATH_ROWS changes.
SEASON_MONTHS = {6, 7, 8}      # winter
SEASON_CENTER = 7              # July — used only for the season-distance tie-break


def _month(dt):
    return int(dt[5:7])


def _season_distance(feat):
    """Circular distance in months from a feature's date to the season center — smaller is closer."""
    d = abs(_month(feat['properties']['datetime']) - SEASON_CENTER)
    return min(d, 12 - d)


def _cloud(feat):
    c = feat['properties'].get('cloud_cover')
    return c if c is not None else 100.0


def _sort_key(feat):
    # primary: cloud cover (lower better); secondary: distance to season center (closer better)
    return (_cloud(feat), _season_distance(feat))


selected_per_pr = {}
exceptions = []      # (path, row, date) — picked, but nothing in-season available
unavailable = []     # path/row with zero candidates at all — not picked, needs attention

for path, row in PATH_ROWS:
    candidates = [f for f in lgi_features
                  if f['properties']['path'] == path and f['properties']['row'] == row]
    if not candidates:
        unavailable.append((path, row))
        continue

    in_season = [f for f in candidates if _month(f['properties']['datetime']) in SEASON_MONTHS]

    if in_season:
        pick = min(in_season, key=_sort_key)
    else:
        pick = min(candidates, key=_sort_key)
        exceptions.append((path, row, pick['properties']['datetime']))

    selected_per_pr[(path, row)] = pick

years = sorted({p['properties']['datetime'][:4] for p in selected_per_pr.values()})

print(f'Selected 1 scene for each of {len(selected_per_pr)}/{len(PATH_ROWS)} path/rows '
      f'(season months={sorted(SEASON_MONTHS)})')
print(f'Year spread across picks: {years}\n')

if unavailable:
    print(f'UNAVAILABLE — zero candidates at all, not selected ({len(unavailable)}):')
    for path, row in unavailable:
        print(f'  {path}/{row}')
    print()

if exceptions:
    print(f'Exceptions — no in-season scene, used nearest date instead ({len(exceptions)}):')
    for path, row, dt in exceptions:
        print(f'  {path}/{row}: {dt}')
    print()

print(f'{"Path/Row":<10} {"Date":<12} {"Cloud%":<8} {"Scene ID"}')
print('-' * 60)
for (path, row), feat in sorted(selected_per_pr.items()):
    p = feat['properties']
    date = p['datetime'][:10]
    cloud = p.get('cloud_cover')
    cloud_str = f'{cloud:.1f}' if cloud is not None else '?'
    print(f'{path}/{row:<7} {date:<12} {cloud_str:<8} {feat["id"]}')

Selected 1 scene for each of 44/44 path/rows (season months=[6, 7, 8])
Year spread across picks: ['2020', '2021', '2022', '2023', '2024', '2025', '2026']

Exceptions — no in-season scene, used nearest date instead (1):
  207/153: 2022-11-24T13:57:49

Path/Row   Date         Cloud%   Scene ID
------------------------------------------------------------
205/151     2023-06-08   0.0      CBERS4A_WPM20515120230608
205/152     2023-07-09   10.0     CBERS4A_WPM20515220230709
206/151     2023-07-04   0.0      CBERS4A_WPM20615120230704
206/152     2023-07-04   0.0      CBERS4A_WPM20615220230704
206/153     2023-07-04   0.0      CBERS4A_WPM20615320230704
207/151     2023-08-30   0.0      CBERS4A_WPM20715120230830
207/152     2025-06-11   0.0      CBERS4A_WPM20715220250611ETC2
207/153     2022-11-24   0.0      CBERS4A_WPM20715320221124
207/154     2026-06-18   10.0     CBERS4A_WPM20715420260618ETC2
207/155     2023-08-30   10.0     CBERS4A_WPM20715520230830
208/149     2026-07-14   0.0      CBER

In [5]:
import geopandas as gpd

selected_gdf = gpd.GeoDataFrame.from_features(
    [
        {
            'type': 'Feature',
            'geometry': feat['geometry'],
            'properties': {
                'path':        p['path'],
                'row':         p['row'],
                'scene_id':    feat['id'],
                'datetime':    p['datetime'],
                'cloud_cover': p.get('cloud_cover'),
                'in_season':   (path, row, p['datetime']) not in
                               {(ep, er, edt) for ep, er, edt in exceptions},
            },
        }
        for (path, row), feat in selected_per_pr.items()
        for p in [feat['properties']]
    ],
    crs='EPSG:4326',
)

out_path = OUT_DIR / 'selected_scenes_pampa44.geojson'
OUT_DIR.mkdir(parents=True, exist_ok=True)
selected_gdf.to_file(out_path, driver='GeoJSON')
print(f'{len(selected_gdf)} selected scenes → {out_path}')
print(f'in_season=False (exceptions): {(~selected_gdf["in_season"]).sum()}')

44 selected scenes → data/inpe/selected_scenes_pampa44.geojson
in_season=False (exceptions): 1


## Step 1d — Tile grid + cross-scene overlap removal

Adjacent scenes overlap significantly (checked: ~26% sidelap along-track between neighbouring
rows, similar cross-track) — naively tiling each scene independently would double-label the
overlap strip. Build a 1024×1024px (2048m at 2m/px) tile grid per scene in its own native UTM
CRS, starting at the scene's top-left corner; partial tiles at the bottom/right edge are
dropped (not padded) since the neighbouring scene's own grid covers that gap.

Selected path/rows straddle both UTM 21S and 22S, so overlap can't be tested in native
per-scene coordinates — reproject a copy of every tile to a common CRS (`EPSG:5880`,
SIRGAS 2000 / Brazil Polyconic, metric) for the cross-scene intersection test only, then
resolve overlaps: a tile is only dropped if it's **fully covered** by the union of its
closer-to-center neighbors, not just on any intersection — a pure "keep whichever tile is
closer to its scene's center" rule discards the loser's non-overlapping sliver too (measured
2.7% of total tile-union area lost this way), since per-scene grids have independent origins
and rarely overlap as a clean full match. Native-CRS geometry preserved for survivors.

Output feeds into QGIS: load `tiles_pampa44.geojson`, Select by Location against the Pampa
polygon, drop tiles with no intersection.

In [6]:
from pystac_client import Client
import rasterio

TILE_PX = 1024
RES_M   = 2.0
TILE_M  = TILE_PX * RES_M   # 2048

bdc_client = Client.open(BDC_STAC_URL)

def selected_gdf_to_bdc_id(row):
    path = str(row['path']).zfill(3)
    row_ = str(row['row']).zfill(3)
    date = row['datetime'][:10].replace('-', '')
    return f'CBERS_4A_WPM_{date}_{path}_{row_}_L4'

# native-CRS raster bounds per scene — reads COG headers only (no pixel data)
scene_bounds = {}
for _, r in selected_gdf.iterrows():
    bdc_id = selected_gdf_to_bdc_id(r)
    items = list(bdc_client.search(collections=[BDC_COLLECTION], ids=[bdc_id]).items())
    if not items:
        print(f'{bdc_id}: not found in BDC — skipping')
        continue
    href = items[0].assets['BAND0'].href
    with rasterio.open(href) as src:
        b = src.bounds
        scene_bounds[bdc_id] = dict(
            path=r['path'], row=r['row'], crs=src.crs.to_string(),
            left=b.left, bottom=b.bottom, right=b.right, top=b.top,
        )

print(f'{len(scene_bounds)}/{len(selected_gdf)} scene bounds resolved')

44/44 scene bounds resolved


In [7]:
from shapely.geometry import box
from shapely.ops import unary_union
import pandas as pd

COMMON_CRS = 'EPSG:5880'   # SIRGAS 2000 / Brazil Polyconic — metric, all-Brazil, for overlap test only

# --- build per-scene tile grid, in each scene's own native CRS ---
tile_rows = []
for scene_id, b in scene_bounds.items():
    left, bottom, right, top = b['left'], b['bottom'], b['right'], b['top']
    cx, cy = (left + right) / 2, (bottom + top) / 2

    nx = int((right - left) // TILE_M)   # full tiles only, from top-left corner —
    ny = int((top - bottom) // TILE_M)   # ragged bottom/right edge dropped, covered by neighbor scene

    for i in range(nx):
        for j in range(ny):
            x0 = left + i * TILE_M
            y1 = top - j * TILE_M
            x1, y0 = x0 + TILE_M, y1 - TILE_M
            tcx, tcy = (x0 + x1) / 2, (y0 + y1) / 2
            dist = ((tcx - cx) ** 2 + (tcy - cy) ** 2) ** 0.5
            tile_rows.append(dict(
                scene_id=scene_id, path=b['path'], row=b['row'], crs=b['crs'],
                dist_to_center=dist, geometry=box(x0, y0, x1, y1),
                tile_id=f'{scene_id}_{i}_{j}',
            ))

print(f'{len(tile_rows)} candidate tiles across {len(scene_bounds)} scenes')

# reproject each scene's tiles (grouped by native CRS) into COMMON_CRS for the overlap test
df = pd.DataFrame(tile_rows)
tiles = pd.concat([
    gpd.GeoDataFrame(sub.drop(columns='crs'), geometry='geometry', crs=crs).to_crs(COMMON_CRS)
    for crs, sub in df.groupby('crs')
], ignore_index=True)
tiles = gpd.GeoDataFrame(tiles, geometry='geometry', crs=COMMON_CRS)

# --- cross-scene overlap resolution ---
# Naive "drop the farther tile on any intersection" loses real coverage: two tiles from
# different scenes are rarely a clean full overlap (grids have independent per-scene
# origins), so the loser's non-overlapping sliver gets discarded too — measured 2.7% of
# total tile-union area lost this way on the full 44-scene set. Fix: only drop a tile if
# it's FULLY covered by the union of its closer-to-center neighbors; if any sliver of it
# sticks out uncovered, keep it regardless of the distance comparison. This guarantees the
# resolution step itself introduces zero coverage gap (only the pre-existing ~0.5% gap from
# the tile grid not tessellating a scene's ragged edge remains, unrelated to this step).
joined = gpd.sjoin(tiles, tiles, how='inner', predicate='intersects', lsuffix='_a', rsuffix='_b')
joined = joined[joined['scene_id__a'] != joined['scene_id__b']]

geom_by_id = dict(zip(tiles['tile_id'], tiles['geometry']))
closer_neighbors = {}   # tile_id -> [ids of intersecting tiles that would win the naive rule]
for a, b_, da, db in zip(joined['tile_id__a'], joined['tile_id__b'],
                          joined['dist_to_center__a'], joined['dist_to_center__b']):
    if da > db or (da == db and a > b_):   # tie-break: deterministic, keep lexicographically smaller
        closer_neighbors.setdefault(a, []).append(b_)

drop_ids = set()
for tid, nbr_ids in closer_neighbors.items():
    covering = unary_union([geom_by_id[n] for n in nbr_ids])
    if geom_by_id[tid].difference(covering).area <= 1e-6:   # fully redundant — safe to drop
        drop_ids.add(tid)

survivors = tiles[~tiles['tile_id'].isin(drop_ids)].copy()
print(f'{len(drop_ids)} tiles dropped as fully redundant, {len(survivors)}/{len(tiles)} survive')

# --- export survivors, reprojected to EPSG:4326, for QGIS Pampa-intersection trim ---
out = survivors.to_crs('EPSG:4326')[['tile_id', 'scene_id', 'path', 'row', 'dist_to_center', 'geometry']]
out_path = OUT_DIR / 'tiles_pampa44.geojson'
out.to_file(out_path, driver='GeoJSON')
print(f'{len(out)} tiles → {out_path}')

139046 candidate tiles across 44 scenes
54117 tiles dropped as fully redundant, 84929/139046 survive
84929 tiles → data/inpe/tiles_pampa44.geojson


## Step 1e — Sample tiles for the hand-labeling budget

Input is `tiles_strictly_covering_pampa.geojson` (QGIS output: Pampa-trimmed via Zonal
Statistics `_count`, see `notes/logbook.md` 2026-07-24 entry) — 49635 tiles across 44 scenes,
scene sizes ranging 9–1809 tiles (Pampa-trim leaves some scenes with only a sliver inside the
biome).

A plain pooled shuffle (previous version of this cell) draws each scene in proportion to its
surviving tile count, but proportion isn't a guarantee — at budget=100 the 9-tile scene has
~98% chance of drawing zero tiles. Each scene is a distinct acquisition (own date, sun angle,
atmosphere, possibly calibration batch), so a scene with zero labeled tiles is an untested
radiometric regime — can't tell class-boundary error from acquisition-specific domain shift
there, and small scenes here are disproportionately biome-edge slivers, i.e. exactly where
that blind spot is worst.

Two-stage allocation, both prefix-stable (extending `SAMPLE_BUDGET` later never changes
earlier picks — required because manually-picked tiles need to be skippable and later budget
top-ups mustn't reshuffle prior choices):

1. **Floor** — every scene gets `FLOOR_R` guaranteed tiles, drawn from its own seeded-shuffled
   pool. Scenes already covered by manual picks (`MANUAL_TILE_IDS`, empty for now) need fewer
   or zero floor tiles — floor is topped up *after* accounting for manual coverage, not on top
   of it.
2. **Remainder** — a sequential divisor method (Sainte-Laguë apportionment, same family used
   for proportional-representation seat allocation) spends the rest of the budget one tile at
   a time, always picking the scene currently furthest below its target share. Target share
   `p_i = (scene's full tile count) / (total tile count)` — uses the *full* count, not the
   post-manual-removal count, so manual picks don't distort what "proportional" means. This is
   append-only by construction (never revisits a past pick), so it's monotonic — no Alabama
   paradox, unlike batch quota + largest-remainder rounding.

Output: `tiles_sample_100.geojson`, footprints of the algorithmically-selected tiles only (not
the manual set, which is tracked separately).</cell id="3ebc6bd6">


In [12]:
import random
import heapq

MANUAL_TILE_IDS = set()   # populate with manually hand-picked tile_ids once that set exists
SAMPLE_SEED     = 42
SAMPLE_BUDGET   = 500
FLOOR_R         = 1       # guaranteed sampled tiles per scene, on top of any manual coverage

pampa_tiles = gpd.read_file(OUT_DIR / 'tiles_strictly_covering_pampa.geojson')

# target share uses each scene's FULL tile count — manual picks must not shrink the goal
scene_total = pampa_tiles.groupby('scene_id').size()
N_total     = len(pampa_tiles)
target_frac = (scene_total / N_total).to_dict()

# per-scene pool of non-manual tiles, shuffled once with a fixed seed -> fixed draw order,
# which is what makes both stages below prefix-stable under a growing budget
rng = random.Random(SAMPLE_SEED)
pools = {}
for sid, sub in pampa_tiles.groupby('scene_id'):
    remaining = [t for t in sub['tile_id'] if t not in MANUAL_TILE_IDS]
    rng.shuffle(remaining)
    pools[sid] = remaining

manual_count = (pampa_tiles[pampa_tiles['tile_id'].isin(MANUAL_TILE_IDS)]
                ['scene_id'].value_counts().to_dict())

# --- stage 1: per-scene floor, topped up only for what manual picks don't already cover ---
cursor   = {sid: 0 for sid in pools}
c        = {sid: 0 for sid in pools}   # tiles counted toward this scene: manual + algorithmic
selected = []

for sid in pools:
    need = max(0, FLOOR_R - manual_count.get(sid, 0))
    take = min(need, len(pools[sid]) - cursor[sid])
    for _ in range(take):
        selected.append(pools[sid][cursor[sid]])
        cursor[sid] += 1
    c[sid] = manual_count.get(sid, 0) + cursor[sid]

print(f'stage 1 (floor={FLOOR_R}): {len(selected)} tiles reserved across {len(pools)} scenes')

# --- stage 2: sequential Sainte-Laguë divisor method spends the rest of the budget ---
# each step: pick the scene maximizing p_i / (2*c_i + 1), i.e. currently furthest below its
# target share; append-only, so extending SAMPLE_BUDGET later never changes earlier picks.
remaining_budget = SAMPLE_BUDGET - len(selected)
heap = []
for sid in pools:
    if cursor[sid] < len(pools[sid]):
        heapq.heappush(heap, (-target_frac[sid] / (2 * c[sid] + 1), sid))

while heap and remaining_budget > 0:
    _, sid = heapq.heappop(heap)
    if cursor[sid] >= len(pools[sid]):
        continue
    selected.append(pools[sid][cursor[sid]])
    cursor[sid] += 1
    c[sid] += 1
    remaining_budget -= 1
    if cursor[sid] < len(pools[sid]):
        heapq.heappush(heap, (-target_frac[sid] / (2 * c[sid] + 1), sid))

# preserve pick order (floor picks first, then divisor draw order), not geojson row order
order = {tid: i for i, tid in enumerate(selected)}
tile_sample = pampa_tiles[pampa_tiles['tile_id'].isin(selected)].copy()
tile_sample['_pick_order'] = tile_sample['tile_id'].map(order)
tile_sample = tile_sample.sort_values('_pick_order').drop(columns='_pick_order')

hit_scenes = tile_sample['scene_id'].nunique()
all_scenes = pampa_tiles['scene_id'].nunique()
print(f'{len(tile_sample)}/{len(pampa_tiles)} tiles sampled (seed={SAMPLE_SEED}, floor={FLOOR_R})')
print(f'{hit_scenes}/{all_scenes} scenes represented in the sample')

missed = set(pampa_tiles['scene_id']) - set(tile_sample['scene_id'])
if missed:
    counts = pampa_tiles['scene_id'].value_counts()
    print(f'\n{len(missed)} scenes with zero sampled tiles (pool size shown for context):')
    for sid in sorted(missed):
        print(f'  {sid}: {counts[sid]} tiles in pool')

out_path = OUT_DIR / f'tiles_sample_{SAMPLE_BUDGET}.geojson'
tile_sample.to_file(out_path, driver='GeoJSON')
print(f'\n-> {out_path}')

stage 1 (floor=1): 44 tiles reserved across 44 scenes
500/49635 tiles sampled (seed=42, floor=1)
44/44 scenes represented in the sample

-> data/inpe/tiles_sample_500.geojson
